In [ ]:
import pandas as pd
import json

java_gt = pd.read_json("../benchmark/final/java_api_update_rules_exact.json")
py_gt = pd.read_json("../benchmark/final/python_api_update_rules_exact.json")

In [ ]:
def pair_accuracy(llm: str, group: str, lang: str):
    gt = {}
    for data in json.load(
        open(f"../benchmark/final/{lang}_api_update_rules_{group}.json")
    ):
        package = data["package"]
        gt[package] = gt.get(package, {})
        old_api = data["old_api"]
        new_api = data["new_api"]
        gt[package][old_api] = [a.split("(")[0] for a in new_api]
    acc = 0
    total = 0
    for data in json.load(open(f"../result/{llm}-{lang}-pair-{group}.json")):
        package = data["package"]
        old_api = data["old_api"]
        output = data["output"].split("(")[0]
        if output in gt[package][old_api]:
            acc += 1
        total += 1

    return acc, total, acc / total


pair_accuracy("gpt-4.1", "exact", "java")

(929, 1385, 0.6707581227436823)

In [22]:
for llm in ["gpt-4.1", "claude-3.7-sonnet", "deepseek-v3", "qwen-plus"]:
    for lang in ["java", "python"]:
        for group in ["exact", "full"]:
            print(llm, group, lang, pair_accuracy(llm, group, lang))

gpt-4.1 exact java (929, 1385, 0.6707581227436823)
gpt-4.1 full java (7600, 21579, 0.3521942629408221)
gpt-4.1 exact python (708, 1603, 0.4416718652526513)
gpt-4.1 full python (3369, 14068, 0.23947967017344327)
claude-3.7-sonnet exact java (990, 1385, 0.7148014440433214)
claude-3.7-sonnet full java (9339, 21589, 0.43258140719811017)
claude-3.7-sonnet exact python (787, 1603, 0.4909544603867748)
claude-3.7-sonnet full python (3814, 14068, 0.27111174296275237)
deepseek-v3 exact java (898, 1385, 0.6483754512635379)
deepseek-v3 full java (7561, 21579, 0.350386950275731)
deepseek-v3 exact python (712, 1603, 0.4441671865252651)
deepseek-v3 full python (3114, 14068, 0.2213534262155246)
qwen-plus exact java (548, 1385, 0.3956678700361011)
qwen-plus full java (4508, 21589, 0.20881004215109547)
qwen-plus exact python (373, 1603, 0.23268870867124142)
qwen-plus full python (1829, 14068, 0.1300113733295422)


In [23]:
from extract_java_snippets import extract_arguments as esjava
from extract_python_snippets import extract_arguments as espython


def instance_accuracy(llm: str, lang: str):
    if lang == "java":
        es = esjava
    elif lang == "python":
        es = espython
    am = 0
    em = 0
    total = 0
    for data in json.load(open(f"../result/{llm}-{lang}-instance-exact.json")):
        gt_args = data["new_args"]
        if "output" not in data:
            continue
        output_args = es(data["output"], data["new_api"])
        total += 1
        if len(output_args) > 0:
            am += 1
            if output_args == gt_args:
                em += 1
    return am, em, total, am / total, em / total


instance_accuracy("gpt-4.1", "java")

(712, 633, 1007, 0.7070506454816285, 0.6285998013902682)

In [25]:
for llm in ["gpt-4.1", "claude-3.7-sonnet", "deepseek-v3", "qwen-plus"]:
    for lang in ["java", "python"]:
        print(llm, lang, instance_accuracy(llm, lang))

gpt-4.1 java (712, 633, 1007, 0.7070506454816285, 0.6285998013902682)
gpt-4.1 python (613, 512, 1016, 0.6033464566929134, 0.5039370078740157)
claude-3.7-sonnet java (589, 553, 1007, 0.5849056603773585, 0.5491559086395233)
claude-3.7-sonnet python (592, 547, 1016, 0.5826771653543307, 0.5383858267716536)
deepseek-v3 java (733, 673, 1004, 0.7300796812749004, 0.6703187250996016)
deepseek-v3 python (609, 532, 1017, 0.5988200589970502, 0.5231071779744346)
qwen-plus java (382, 344, 1006, 0.3797216699801193, 0.341948310139165)
qwen-plus python (536, 456, 1014, 0.5285996055226825, 0.44970414201183434)
